# simplicio-video · render no Colab (GPU)

Renderiza um **bundle** gerado por `simplicio-video render <contrato.yaml> --backend colab` e devolve o MP4.
Imprime o tempo de cada etapa (setup, frames, encode, master) e a GPU usada, para comparar com o render local
(o `bundle.json` traz os hashes e tempos do render local de referência quando existem).

1. **Ambiente de execução → Alterar tipo → GPU** (T4 do plano gratuito basta). Nada aqui cria conta ou gasta créditos.
2. Rode as células na ordem. A célula 2 detecta o Ubuntu da imagem do Colab, instala só o que falta e **mostra o erro real** se algo falhar.
3. Bundle: **upload** na célula 4 (padrão) **ou** rode a célula 3 (opcional) para montar o Drive e usar
   `BUNDLE = "/content/drive/MyDrive/simplicio-video/colab/<bundle>.zip"`.
4. A última célula baixa o MP4. Suba no Drive em `drive.caminho` do contrato (pasta `simplicio-video/clientes/...`).

Chrome: usa o **chrome-headless-shell fixado pelo HyperFrames** (mesma build do render local), instalado com
`@puppeteer/browsers --install-deps`, que resolve as libs do sistema pelo `deb.deps` do próprio Chrome
(ex.: `libasound2` ou `libasound2t64` no Ubuntu 24.04). Se isso falhar, a célula instala lib por lib com alternativas.

Determinismo: com o mesmo Chrome/HyperFrames e encode por CPU, os frames saem idênticos aos locais; com NVENC (`--gpu`)
o vídeo é recodificado por outro encoder, então os bytes mudam (o áudio masterizado deve bater).

In [ ]:
# 1) GPU, relógio e helpers (erros aparecem com stdout/stderr reais)
import glob, json, os, re, subprocess, time, zipfile
T = {}
T0 = time.time()

def run(cmd, env=None):
    """Run a shell command and return the CompletedProcess (never raises)."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)

def sh(cmd, env=None):
    """Run a shell command; on failure raise with exit code + real stdout/stderr (no hidden errors)."""
    p = run(cmd, env)
    if p.returncode != 0:
        raise RuntimeError(f"falhou (exit {p.returncode}): {cmd}\n--- stdout ---\n{p.stdout[-3000:]}\n--- stderr ---\n{p.stderr[-3000:]}")
    return p.stdout

gpu = run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader").stdout.strip()
print("GPU:", gpu or "nenhuma (runtime CPU)")

In [ ]:
# 2) Setup robusto: Ubuntu detectado, Node 22, ffmpeg (pula se já existe), Chrome do HyperFrames + deps
t = time.time()
NODE = "v22.23.3"                    # mesma versão do box
HF = "hyperframes@0.8.97"            # fixado (mesmo do CLI)
PB = "@puppeteer/browsers@3.2.3"     # mesma versão que o HyperFrames 0.8.97 usa
CHROME_FALLBACK = "152.0.7977.30"    # CHROME_VERSION do HyperFrames 0.8.97
APT_ENV = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}

osr = dict(l.split("=", 1) for l in open("/etc/os-release").read().splitlines() if "=" in l)
UBU = osr.get("VERSION_ID", "0").strip('"')
UBU_MAJOR = int(UBU.split(".")[0] or 0)
print("SO:", osr.get("PRETTY_NAME", "?").strip('"'), "| root:", os.geteuid() == 0)

_apt_updated = False
def apt_update():
    global _apt_updated
    if _apt_updated:
        return
    p = run("apt-get update", APT_ENV)   # a repo de terceiros quebrada não deve derrubar o setup: mostra e segue
    if p.returncode != 0:
        print(f"aviso: apt-get update exit {p.returncode} (seguindo com as listas atuais)\n{p.stderr[-1500:]}")
    _apt_updated = True

def installed(pkg):
    return "install ok installed" in run(f"dpkg-query -W -f='${{Status}}' {pkg}").stdout

def candidate(pkg):
    m = re.search(r"Candidate:\s*(\S+)", run(f"apt-cache policy {pkg}").stdout)
    return bool(m) and m.group(1) != "(none)"

def apt_install_any(groups):
    """groups = [[alternativa1, alternativa2, ...], ...]; instala a primeira alternativa disponível de cada grupo."""
    apt_update()
    todo, missing = [], []
    for alts in groups:
        alts = sorted(alts, key=lambda p: (UBU_MAJOR >= 24) != p.endswith("t64"))  # 24.04+: prefere *t64
        if any(installed(p) for p in alts):
            continue
        pick = next((p for p in alts if candidate(p)), None)
        (todo.append(pick) if pick else missing.append(" | ".join(alts)))
    if missing:
        print("aviso: sem pacote disponível para:", "; ".join(missing))
    if not todo:
        return
    print("apt-get install:", " ".join(todo))
    p = run("apt-get install -y --no-install-recommends " + " ".join(todo), APT_ENV)
    if p.returncode != 0:  # descobre qual pacote quebra e mostra o erro real
        print(f"apt-get install em lote falhou (exit {p.returncode}):\n{p.stderr[-2000:]}\nTentando um por um...")
        bad = []
        for pkg in todo:
            q = run(f"apt-get install -y --no-install-recommends {pkg}", APT_ENV)
            if q.returncode != 0:
                bad.append(pkg); print(f"  {pkg}: exit {q.returncode}\n{q.stderr[-800:]}")
        if bad:
            print("aviso: falharam:", " ".join(bad))

# Node 22
if not run("node --version").stdout.startswith("v22"):
    sh(f"curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1")
print("node", sh("node --version").strip())

# ffmpeg: a imagem do Colab já traz; só instala se faltar
if run("command -v ffmpeg").returncode != 0:
    apt_install_any([["ffmpeg"]])
print(sh("ffmpeg -version").splitlines()[0])
nvenc = "h264_nvenc" in sh("ffmpeg -hide_banner -encoders")
print("NVENC no ffmpeg:", nvenc)

# HyperFrames (baixa o pacote) + versão do Chrome que ele fixa
print(sh(f"npx --yes {HF} --version").strip())
CHROME_VER = CHROME_FALLBACK
for f in glob.glob(os.path.expanduser("~/.npm/_npx/*/node_modules/hyperframes/dist/*.js")):
    m = re.search(r'var CHROME_VERSION = "([0-9.]+)"', open(f, encoding="utf8", errors="ignore").read())
    if m:
        CHROME_VER = m.group(1); break
CACHE = os.path.expanduser("~/.cache/hyperframes/chrome")   # cache do HyperFrames (layout do @puppeteer/browsers)

# Chrome headless do HyperFrames + libs do sistema pelo deb.deps do próprio Chrome (apt-get satisfy)
p = run(f"npx --yes {PB} install chrome-headless-shell@{CHROME_VER} --path {CACHE} --install-deps --format '{{{{path}}}}'")
if p.returncode == 0:
    CHROME = p.stdout.strip().splitlines()[-1]
else:
    err = (p.stderr[p.stderr.find("Error:"):] if "Error:" in p.stderr else p.stderr)[-2000:]   # sem o texto de ajuda do yargs
    print(f"aviso: puppeteer --install-deps falhou (exit {p.returncode}):\n{err}\n"
          "Baixando o Chrome sem deps e instalando as libs uma a uma...")
    CHROME = sh(f"npx --yes {PB} install chrome-headless-shell@{CHROME_VER} --path {CACHE} --format '{{{{path}}}}'").strip().splitlines()[-1]

CHROME_DEPS = [["libasound2t64", "libasound2"], ["libatk-bridge2.0-0t64", "libatk-bridge2.0-0"], ["libatk1.0-0t64", "libatk1.0-0"],
               ["libatspi2.0-0t64", "libatspi2.0-0"], ["libcups2t64", "libcups2"], ["libglib2.0-0t64", "libglib2.0-0"],
               ["libgtk-3-0t64", "libgtk-3-0"], ["libgbm1"], ["libnss3"], ["libnspr4"], ["libpango-1.0-0"], ["libcairo2"],
               ["libxkbcommon0"], ["libxcomposite1"], ["libxdamage1"], ["libxrandr2"], ["libxfixes3"], ["libxext6"], ["libx11-6"],
               ["libxcb1"], ["libdbus-1-3"], ["libexpat1"], ["libudev1"], ["libvulkan1"], ["fonts-liberation"], ["ca-certificates"]]
missing_so = lambda: sorted(set(re.findall(r"^\s*(\S+) => not found", run(f"ldd '{CHROME}'").stdout, re.M)))
if (p.returncode != 0 and os.geteuid() == 0) or missing_so():
    apt_install_any(CHROME_DEPS)
if missing_so():
    raise RuntimeError(f"Chrome ainda sem libs: {missing_so()} (Ubuntu {UBU}). Veja os avisos do apt acima.")
os.environ["HYPERFRAMES_BROWSER_PATH"] = CHROME   # render_bundle.sh usa exatamente este Chrome
print("chrome:", CHROME_VER, CHROME)
print(sh(f"'{CHROME}' --version").strip())
d = run(f"npx --yes {HF} doctor")
print(d.stdout[-1500:] or d.stderr[-1500:])
T["setup_s"] = round(time.time() - t, 1)
print("setup:", T["setup_s"], "s")

In [ ]:
# 3) (Opcional) Drive: monte e aponte BUNDLE para o zip no Drive. Pule esta célula para usar upload na célula 4.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_COLAB = "/content/drive/MyDrive/simplicio-video/colab"
BUNDLE = f"{DRIVE_COLAB}/<bundle>.zip"   # troque <bundle> pelo nome do zip; se houver um único zip na pasta, ele é usado
print("\n".join(sorted(glob.glob(f"{DRIVE_COLAB}/*.zip"))) or f"nenhum .zip em {DRIVE_COLAB}")

In [ ]:
# 4) Bundle: upload (padrão) ou BUNDLE vindo da célula 3 (arquivo .zip ou pasta com zips)
BUNDLE = globals().get("BUNDLE", "")
if BUNDLE and "<bundle>" in BUNDLE:
    zips = sorted(glob.glob(os.path.join(os.path.dirname(BUNDLE), "*.zip")))
    if len(zips) != 1:
        raise FileNotFoundError(f"troque <bundle> em BUNDLE pelo nome do zip; encontrados: {zips or 'nenhum'}")
    BUNDLE = zips[0]
if BUNDLE and os.path.isdir(BUNDLE):
    zips = sorted(glob.glob(os.path.join(BUNDLE, "*.zip")), key=os.path.getmtime)
    if not zips:
        raise FileNotFoundError(f"nenhum .zip em {BUNDLE}")
    BUNDLE = zips[-1]
if not BUNDLE:
    from google.colab import files
    up = files.upload()
    BUNDLE = os.path.abspath(next(iter(up)))
if not os.path.isfile(BUNDLE):
    raise FileNotFoundError(f"BUNDLE não encontrado: {BUNDLE}")
print("bundle:", BUNDLE)
sh("rm -rf /content/bundle && mkdir -p /content/bundle")
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall("/content/bundle")
meta = json.load(open("/content/bundle/bundle.json"))
print(json.dumps({k: meta.get(k) for k in ("contract", "inputs_sha256", "quality", "output_name")}, indent=1, ensure_ascii=False))

In [ ]:
# 5) Render (frames + encode) + master -14 LUFS; GPU encode quando houver NVENC
t = time.time()
p = run("cd /content/bundle && bash render_bundle.sh", {**os.environ, "HYPERFRAMES_BROWSER_PATH": CHROME})
print(p.stdout[-2500:], p.stderr[-2500:])
if p.returncode != 0:
    raise RuntimeError(f"render falhou (exit {p.returncode}); veja stdout/stderr acima e /content/bundle/out/render.log")
tim = json.loads(re.search(r"TIMINGS (\{.*\})", p.stdout).group(1))
T.update({"frames_capture_s": tim["capture_s"], "encode_s": tim["encode_s"], "render_total_s": tim["render_s"],
          "master_s": tim["master_s"], "gpu_encode": tim["gpu_encode"]})
T["total_s"] = round(time.time() - T0, 1)

In [ ]:
# 6) Tempos por etapa + comparação com o render local de referência
import hashlib
out = "/content/bundle/out/final.mp4"
dec = lambda a: subprocess.run(f"ffmpeg -v error -i {out} {a} -", shell=True, capture_output=True).stdout
frames = [l.rsplit(",", 1)[-1].strip() for l in dec("-map 0:v:0 -f framemd5").decode().splitlines() if l and not l.startswith("#")]
res = {"gpu": gpu or "cpu", "chrome": CHROME_VER, **T, "frames": len(frames),
       "video_frames_sha256": hashlib.sha256("\n".join(frames).encode()).hexdigest(),
       "audio_pcm_sha256": hashlib.sha256(dec("-map 0:a:0 -f s16le -ac 2 -ar 48000")).hexdigest()}
ref = meta.get("reference_local")
if ref:
    res["local_timings_s"] = ref.get("timings_s")
    res["frames_iguais_ao_local"] = res["video_frames_sha256"] == ref["video_frames_sha256"]
    res["audio_igual_ao_local"] = res["audio_pcm_sha256"] == ref["audio_pcm_sha256"]
print(json.dumps(res, indent=1, ensure_ascii=False))

In [ ]:
# 7) Baixar o MP4 (nome do contrato). Upload no Drive: pasta drive.caminho do contrato.
from google.colab import files
dst = "/content/" + meta["output_name"]
sh(f"cp {out} '{dst}'")
print("Drive:", meta["drive"]["caminho"], "->", meta["drive"]["arquivo"])
files.download(dst)